In [2]:
import os
import sqlite3
import pandas as pd
import warnings
from loguru import logger

warnings.filterwarnings('ignore')

# Optioneel: Sla de logs ook op in een bestand
logger.add("sdm_pipeline.log", rotation="1 MB")


1

  Opdracht 5

  Opdracht 5.a







  Reset, Create en Load Database functies

In [3]:

def reset_sdm():
    with sqlite3.connect('sdm.db') as conn:
        tables = pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
        cursor = conn.cursor()
        for table in tables['name']:
            cursor.execute(f"DROP TABLE IF EXISTS {table}")
        conn.commit()
    logger.success("Database succesvol geleegd.")


def create_sdm_tables():
    sql_file = 'BikeToDrive_RIM - alle bestanden.txt'
    if not os.path.exists(sql_file):
        logger.warning(f"Let op: {sql_file} niet gevonden!")
        return

    with open(sql_file, 'r', encoding='utf-8') as f:
        sql = f.read()

    with sqlite3.connect('sdm.db') as sdm_conn:
        cursor = sdm_conn.cursor()
        statements = [stmt.strip() for stmt in sql.split(
            ';') if stmt.strip() and not stmt.strip().startswith('--')]

        for stmt in statements:
            try:
                cursor.execute(stmt)
            except Exception as e:
                logger.error(f"Error bij uitvoeren DDL: {e} voor {stmt[:50]}...")
        sdm_conn.commit()
    logger.success("Tabellen succesvol aangemaakt in SDM.")


def load_data_to_sdm():
    mappings = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Accessoire_Verkoop': 'Accessoireverkoop',
            'Monteur': 'Accessoireverkoop_Monteur',
            'Leverancier': 'Accessoireverkoop_Leverancier',
            'Accessoire': 'Accessoireverkoop_Accessoire',
            'Filiaal': 'Accessoireverkoop_filiaal',
            'Klant': 'Accessoireverkoop_klant'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Fiets_Verkoop': 'Fietsverkoop',
            'Fiets': 'Fietsverkoop_Fiets',
            'Monteur': 'Fietsverkoop_Monteur',
            'Fabrikant': 'Fietsverkoop_Fabrikant',
            'Filiaal': 'Fietsverkoop_filiaal',
            'Klant': 'Fietsverkoop_klant'
        },
        'BikeToDrive_3_Onderhoud.db': {
            'Onderhoud': 'Onderhoud',
            'Fiets': 'Onderhoud_Fiets',
            'Monteur': 'Onderhoud_Monteur',
            'Fabrikant': 'Onderhoud_Fabrikant',
            'Filiaal': 'Onderhoud_filiaal'
        },
        'BikeToDrive_4_Accessoire_Inkoop.db': {
            'Accessoire_Inkoop': 'Accessoire_inkoop',
            'Accessoire': 'Accessoire_inkoop_Accessoire',
            'Leverancier': 'Accessoire_inkoop_Leverancier'
        },
        'BikeToDrive_5_Fiets_Inkoop.db': {
            'Fiets_Inkoop': 'Fiets_Inkoop',
            'Fiets': 'Fiets_Inkoop_Fiets',
            'Fabrikant': 'Fiets_Inkoop_Fabrikant'
        }
    }

    with sqlite3.connect('sdm.db') as sdm_conn:
        for db, table_map in mappings.items():
            if not os.path.exists(db):
                logger.warning(f"Bron DB '{db}' niet gevonden. Wordt overgeslagen.")
                continue

            with sqlite3.connect(db) as conn:
                for src_table, dst_table in table_map.items():
                    try:
                        df = pd.read_sql(f"SELECT * FROM {src_table}", conn)
                        df.to_sql(dst_table, sdm_conn,
                                  if_exists='append', index=False)
                        logger.info(
                            f"Geladen: {len(df)} rijen van {src_table} -> {dst_table}")
                    except Exception as e:
                        logger.error(f"Fout bij inladen {src_table}: {e}")

    logger.success("Data laden completed.")


def full_refresh_sdm():
    logger.info("Start Full Refresh...")
    reset_sdm()
    create_sdm_tables()
    load_data_to_sdm()
    logger.success("Full refresh van SDM completed.")



  Opdracht 5.b



  Database testen en exploreren

In [4]:
dbs = [
    'BikeToDrive_1_Accessoireverkoop.db',
    'BikeToDrive_2_Fietsverkoop.db',
    'BikeToDrive_3_Onderhoud.db',
    'BikeToDrive_4_Accessoire_Inkoop.db',
    'BikeToDrive_5_Fiets_Inkoop.db'
]

for db in dbs:
    if os.path.exists(db):
        with sqlite3.connect(db) as conn:
            tables = pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
            logger.info(f"{db}: {list(tables['name'])}")
    else:
        logger.warning(f"{db} not found")



2026-04-09 14:10:22.236 | INFO     | __main__:<module>:14 - BikeToDrive_1_Accessoireverkoop.db: ['Accessoire_Verkoop', 'Monteur', 'Leverancier', 'Accessoire', 'Filiaal', 'Klant']
2026-04-09 14:10:22.240 | INFO     | __main__:<module>:14 - BikeToDrive_2_Fietsverkoop.db: ['Fiets_Verkoop', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal', 'Klant']
2026-04-09 14:10:22.242 | INFO     | __main__:<module>:14 - BikeToDrive_3_Onderhoud.db: ['Onderhoud', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal']
2026-04-09 14:10:22.244 | INFO     | __main__:<module>:14 - BikeToDrive_4_Accessoire_Inkoop.db: ['Accessoire_Inkoop', 'Accessoire', 'Leverancier']
2026-04-09 14:10:22.245 | INFO     | __main__:<module>:14 - BikeToDrive_5_Fiets_Inkoop.db: ['Fiets_Inkoop', 'Fiets', 'Fabrikant']


In [5]:
shared_tables = ['Klant', 'Monteur', 'Filiaal',
                 'Leverancier', 'Accessoire', 'Fiets', 'Fabrikant']

logger.info("--- Aantal unieke ID's in gedeelde tabellen over de databases ---")
for table in shared_tables:
    ids = set()
    for db in dbs:
        if not os.path.exists(db):
            continue
        try:
            with sqlite3.connect(db) as conn:
                df = pd.read_sql(f"SELECT * FROM {table}", conn)
                pk_col = df.columns[0]
                ids.update(df[pk_col].tolist())
        except:
            pass
    logger.info(f"{table}: {len(ids)} unique IDs gevonden")



2026-04-09 14:10:22.254 | INFO     | __main__:<module>:4 - --- Aantal unieke ID's in gedeelde tabellen over de databases ---
2026-04-09 14:10:22.258 | INFO     | __main__:<module>:17 - Klant: 26 unique IDs gevonden
2026-04-09 14:10:22.262 | INFO     | __main__:<module>:17 - Monteur: 15 unique IDs gevonden
2026-04-09 14:10:22.267 | INFO     | __main__:<module>:17 - Filiaal: 5 unique IDs gevonden
2026-04-09 14:10:22.271 | INFO     | __main__:<module>:17 - Leverancier: 5 unique IDs gevonden
2026-04-09 14:10:22.274 | INFO     | __main__:<module>:17 - Accessoire: 13 unique IDs gevonden
2026-04-09 14:10:22.279 | INFO     | __main__:<module>:17 - Fiets: 75 unique IDs gevonden
2026-04-09 14:10:22.282 | INFO     | __main__:<module>:17 - Fabrikant: 11 unique IDs gevonden


In [6]:
if os.path.exists('BikeToDrive_1_Accessoireverkoop.db'):
    with sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db') as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(Klant)")
        columns = cursor.fetchall()
        logger.info(f"Klant columns in Accessoireverkoop: {[col[1] for col in columns]}")
        df = pd.read_sql("SELECT * FROM Klant LIMIT 5", conn)
        logger.info(f"Data preview Klant:\n{df.head()}")



2026-04-09 14:10:22.289 | INFO     | __main__:<module>:6 - Klant columns in Accessoireverkoop: ['klantnr', 'naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']
2026-04-09 14:10:22.297 | INFO     | __main__:<module>:8 - Data preview Klant:
   klantnr           naam           adres               woonplaats geslacht  \
0        1     Jan Jansen   Kerkstraat 12  Gewijzigde Test Locatie        M   
1        3  Pieter Visser   Havenstraat 3                 Den Haag        M   
2        4      Emma Smit    Boomgaard 22                  Haarlem        V   
3        5     Tom Bakker  Stationsweg 44                   Leiden        M   
4        6    Lisa Meijer   Dijkstraat 10                  Zaandam        V   

  geboortedatum  
0    1985-03-22  
1    1978-11-05  
2    1995-02-18  
3    1982-09-09  
4    1993-12-30  


  Opdracht 6 - Mutaties en CDC Testing

In [7]:
bron_db = 'BikeToDrive_1_Accessoireverkoop.db'
PK_KOLOM = 'klantnr'

if os.path.exists(bron_db):
    with sqlite3.connect(bron_db) as conn_bron:
        cursor_bron = conn_bron.cursor()

        try:
            # 1. INSERT
            cursor_bron.execute(
                f"INSERT INTO Klant ({PK_KOLOM}, naam, woonplaats) VALUES (9999, 'Test Klant Opdracht 6', 'SDM-Stad')")

            # 2. UPDATE
            cursor_bron.execute(
                f"UPDATE Klant SET woonplaats = 'Gewijzigde Test Locatie' WHERE {PK_KOLOM} = 1")

            # 3. DELETE
            cursor_bron.execute(f"DELETE FROM Klant WHERE {PK_KOLOM} = 2")

            conn_bron.commit()
            logger.success("Stap 1: INSERT, UPDATE en DELETE succesvol uitgevoerd op de bron-database.")

        except Exception as e:
            logger.error(f"Er ging iets mis in de brondatabase. Foutmelding: {e}")
            conn_bron.rollback()
else:
    logger.warning(f"Brondatabase {bron_db} niet gevonden voor Opdracht 6.")


logger.info("Starten met Full Refresh van het SDM om wijzigingen op te halen...")
full_refresh_sdm()
logger.success("Stap 2: SDM succesvol opnieuw opgebouwd en geladen met de nieuwste data.")


logger.info("--- Test Resultaten in SDM (Tabel: Accessoireverkoop_klant) ---")
with sqlite3.connect('sdm.db') as sdm_connection:
    try:
        df_insert = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 9999", sdm_connection)
        logger.info("1. INSERT test (Is de nieuwe klant 9999 aanwezig?):")
        if not df_insert.empty:
            logger.success(f"Succes! Klant gevonden:\n{df_insert}")
        else:
            logger.error("Gefaald. Klant niet gevonden.")

        df_update = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 1", sdm_connection)
        logger.info("2. UPDATE test (Heeft klant 1 de nieuwe woonplaats?):")
        if not df_update.empty:
            logger.success(f"Controleer de onderstaande data op de wijziging (Gewijzigde Test Locatie):\n{df_update}")
        else:
            logger.error("Gefaald. Klant 1 bestaat niet in SDM.")

        df_delete = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 2", sdm_connection)
        logger.info("3. DELETE test (Is klant 2 succesvol verwijderd uit SDM?):")
        if df_delete.empty:
            logger.success("Succes! Klant 2 bestaat niet meer in het SDM.")
        else:
            logger.error("Gefaald. Klant 2 is nog steeds aanwezig.")

    except Exception as e:
        logger.error(f"Er is een fout opgetreden bij het valideren van het SDM: {e}")

2026-04-09 14:10:22.313 | SUCCESS  | __main__:<module>:21 - Stap 1: INSERT, UPDATE en DELETE succesvol uitgevoerd op de bron-database.
2026-04-09 14:10:22.314 | INFO     | __main__:<module>:30 - Starten met Full Refresh van het SDM om wijzigingen op te halen...
2026-04-09 14:10:22.315 | INFO     | __main__:full_refresh_sdm:93 - Start Full Refresh...
2026-04-09 14:10:22.422 | SUCCESS  | __main__:reset_sdm:9 - Database succesvol geleegd.
2026-04-09 14:10:22.512 | SUCCESS  | __main__:create_sdm_tables:32 - Tabellen succesvol aangemaakt in SDM.
2026-04-09 14:10:22.519 | INFO     | __main__:load_data_to_sdm:84 - Geladen: 100 rijen van Accessoire_Verkoop -> Accessoireverkoop
2026-04-09 14:10:22.525 | INFO     | __main__:load_data_to_sdm:84 - Geladen: 10 rijen van Monteur -> Accessoireverkoop_Monteur
2026-04-09 14:10:22.532 | INFO     | __main__:load_data_to_sdm:84 - Geladen: 5 rijen van Leverancier -> Accessoireverkoop_Leverancier
2026-04-09 14:10:22.538 | INFO     | __main__:load_data_to_sd